In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt

In [13]:
# Step 1: Download historical stock price data
stock_data = yf.download('AAPL', start='2023-01-01', end='2024-10-23')

[*********************100%***********************]  1 of 1 completed


In [14]:
# Step 2: Calculate daily returns
stock_data['Return'] = stock_data['Adj Close'].pct_change()

# Step 3: Prepare the feature set (using returns and rolling volatility as features)
stock_data['Volatility'] = stock_data['Return'].rolling(window=20).std()
stock_data.dropna(inplace=True)  # Drop NaN values
X = stock_data[['Return', 'Volatility']]

In [40]:
# Step 4: Apply DBSCAN for anomaly detection
dbscan = DBSCAN(eps=0.005, min_samples=7)
clusters = dbscan.fit_predict(X)

In [41]:
# Step 5: Add cluster labels to the dataframe
stock_data['Cluster'] = clusters

In [42]:
# Step 6: Visualize the results
import plotly.express as px

fig = px.scatter(stock_data, x=stock_data.index, y='Adj Close', color='Cluster', title='S&P 500 Stock Price with DBSCAN Clusters')
fig.update_layout(xaxis_title='Date', yaxis_title='Price', width=1000, height=600)
fig.show()


import plotly.graph_objects as go

# Create a figure
fig = go.Figure()

anomalies = stock_data[stock_data['Cluster'] == -1]

# Add the stock price line
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['Adj Close'], mode='lines', name='AAPL Price', line=dict(color='blue')))

# Add the anomalies as scatter points
fig.add_trace(go.Scatter(x=anomalies.index, y=anomalies['Adj Close'], mode='markers', name='Anomalies', marker=dict(color='red', symbol='x')))

# Update layout
fig.update_layout(
    title='AAPL Stock Price with Detected Anomalies (Outliers)',
    xaxis_title='Date',
    yaxis_title='Price',
    width=1000,
    height=600
)

# Show the figure
fig.show()


In [52]:
# Calculate technical indicators: RSI, MACD, and Moving Averages
def calculate_rsi(data, window=14):
    delta = data['Adj Close'].diff(1)
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    avg_gain = gain.rolling(window=window).mean()
    avg_loss = loss.rolling(window=window).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

In [53]:
def calculate_macd(data, short_window=12, long_window=26, signal_window=9):
    short_ema = data['Adj Close'].ewm(span=short_window, adjust=False).mean()
    long_ema = data['Adj Close'].ewm(span=long_window, adjust=False).mean()
    macd = short_ema - long_ema
    signal = macd.ewm(span=signal_window, adjust=False).mean()
    return macd, signal

In [54]:
def calculate_moving_average(data, window=20):
    return data['Adj Close'].rolling(window=window).mean()

In [55]:
# Calculate RSI, MACD and 20-day moving average
stock_data['RSI'] = calculate_rsi(stock_data)
stock_data['MACD'], stock_data['MACD_Signal'] = calculate_macd(stock_data)
stock_data['SMA'] = calculate_moving_average(stock_data)


In [56]:
# Drop rows with NaN values due to rolling windows
stock_data.dropna(inplace=True)

# Prepare features for DBSCAN: RSI, MACD, and Moving Averages
features = stock_data[['RSI', 'MACD', 'SMA']].values

# Apply DBSCAN to cluster the technical indicators
dbscan = DBSCAN(eps=3, min_samples=5)
stock_data['Cluster'] = dbscan.fit_predict(features)

In [57]:
from plotly.subplots import make_subplots

import plotly.graph_objects as go

# Create a subplot figure with 2 rows
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, subplot_titles=('Adjusted Close Price with DBSCAN Clusters', 'RSI and MACD Indicators'))

# Plot Adjusted Close price with clusters
for cluster in stock_data['Cluster'].unique():
    cluster_data = stock_data[stock_data['Cluster'] == cluster]
    fig.add_trace(go.Scatter(x=cluster_data.index, y=cluster_data['Adj Close'], mode='markers', name=f'Cluster {cluster}'), row=1, col=1)

fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['Adj Close'], mode='lines', name='Adj Close', line=dict(color='blue')), row=1, col=1)

# Plot RSI and MACD
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['RSI'], mode='lines', name='RSI', line=dict(color='red')), row=2, col=1)
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['MACD'], mode='lines', name='MACD', line=dict(color='green')), row=2, col=1)
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['MACD_Signal'], mode='lines', name='MACD Signal', line=dict(color='purple')), row=2, col=1)

# Update layout
fig.update_layout(height=800, width=1000, title_text='Stock Data Analysis with DBSCAN Clusters')
fig.update_xaxes(title_text='Date', row=2, col=1)
fig.update_yaxes(title_text='Price', row=1, col=1)
fig.update_yaxes(title_text='Indicator Value', row=2, col=1)

# Show the figure
fig.show()


In [58]:
# Calculate technical indicators: RSI, MACD, and Moving Averages
def calculate_rsi(data, window=14):
    delta = data['Adj Close'].diff(1)
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    avg_gain = gain.rolling(window=window).mean()
    avg_loss = loss.rolling(window=window).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def calculate_macd(data, short_window=12, long_window=26, signal_window=9):
    short_ema = data['Adj Close'].ewm(span=short_window, adjust=False).mean()
    long_ema = data['Adj Close'].ewm(span=long_window, adjust=False).mean()
    macd = short_ema - long_ema
    signal = macd.ewm(span=signal_window, adjust=False).mean()
    return macd, signal

def calculate_moving_average(data, window=20):
    return data['Adj Close'].rolling(window=window).mean()

In [59]:
# Calculate RSI, MACD and 20-day moving average
stock_data['RSI'] = calculate_rsi(stock_data)
stock_data['MACD'], stock_data['MACD_Signal'] = calculate_macd(stock_data)
stock_data['SMA'] = calculate_moving_average(stock_data)

# Drop rows with NaN values due to rolling windows
stock_data.dropna(inplace=True)

# Prepare features for DBSCAN: RSI, MACD, and Moving Averages
features = stock_data[['RSI', 'MACD', 'SMA']].values

# Apply DBSCAN to cluster the technical indicators
dbscan = DBSCAN(eps=3, min_samples=5)
stock_data['Cluster'] = dbscan.fit_predict(features)

# Define buy and sell conditions based on the user's strategy
stock_data['Trend'] = stock_data['Adj Close'].diff().fillna(0)
stock_data['Buy_Signal'] = np.nan
stock_data['Sell_Signal'] = np.nan

In [61]:
import warnings
warnings.filterwarnings("ignore")

# Iterate over the dataset to check for 3 consecutive points in the same cluster
for i in range(4, len(stock_data)):
    # Check if 3 consecutive points are in the same cluster
    if (stock_data['Cluster'].iloc[i] == stock_data['Cluster'].iloc[i-1] == stock_data['Cluster'].iloc[i-2] == stock_data['Cluster'].iloc[i-3]== stock_data['Cluster'].iloc[i-4]):
        # Check if the trend is upward (prices increasing)
        if (stock_data['Adj Close'].iloc[i] > stock_data['Adj Close'].iloc[i-4]):
            stock_data['Buy_Signal'].iloc[i] = stock_data['Adj Close'].iloc[i]
        # Check if the trend is downward (prices decreasing)
        elif (stock_data['Adj Close'].iloc[i] < stock_data['Adj Close'].iloc[i-4]):
            stock_data['Sell_Signal'].iloc[i] = stock_data['Adj Close'].iloc[i]

In [63]:
from plotly.subplots import make_subplots

import plotly.express as px
import plotly.graph_objects as go

# Create a Plotly figure with subplots
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    vertical_spacing=0.1, subplot_titles=('NVIDIA Adjusted Close with Buy/Sell Signals and Clusters', 'RSI and MACD Indicators'))

# Plot Adjusted Close Price
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['Adj Close'], mode='lines', name='Adj Close', line=dict(color='blue')), row=1, col=1)

# Plot Buy and Sell Signals
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['Buy_Signal'], mode='markers', name='Buy Signal', marker=dict(color='green', symbol='triangle-up', size=10)), row=1, col=1)
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['Sell_Signal'], mode='markers', name='Sell Signal', marker=dict(color='red', symbol='triangle-down', size=10)), row=1, col=1)

# Plot the DBSCAN Clusters (with different colors for each cluster)
for cluster in stock_data['Cluster'].unique():
    cluster_data = stock_data[stock_data['Cluster'] == cluster]
    fig.add_trace(go.Scatter(x=cluster_data.index, y=cluster_data['Adj Close'], mode='markers', name=f'Cluster {cluster}'), row=1, col=1)

# Plot RSI
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['RSI'], mode='lines', name='RSI', line=dict(color='red')), row=2, col=1)

# Plot MACD and Signal Line
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['MACD'], mode='lines', name='MACD', line=dict(color='green')), row=2, col=1)
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['MACD_Signal'], mode='lines', name='MACD Signal', line=dict(color='purple')), row=2, col=1)

# Update layout to improve aesthetics
fig.update_layout(height=800, width=1200, title_text='Interactive NVIDIA Trading Strategy with DBSCAN Clusters, Buy/Sell Signals', xaxis_rangeslider_visible=False)

# Show the plot
fig.show()